In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. ĐỌC DỮ LIỆU
df = pd.read_csv('Diab_pyth_data_clean.csv')

# 2. CHUẨN BỊ 2 BIẾN MỤC TIÊU
# Cho Hồi quy (Regression): Giữ nguyên số liên tục
y_reg = df['glyhb'] 
# Cho Phân lớp (Classification): >= 6.5 là bệnh (1)
y_class = df['glyhb'].apply(lambda x: 1 if x >= 6.5 else 0)

# 3. CHUẨN BỊ ĐẦU VÀO (X)
X = df.drop(columns=['glyhb'])
if 'diabetes_status' in X.columns:
    X = X.drop(columns=['diabetes_status'])
if 'diabetes' in X.columns:
    X = X.drop(columns=['diabetes'])

X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

# 4. CHUẨN HÓA DỮ LIỆU (Bắt buộc cho mô hình tuyến tính)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_encoded), columns=X_encoded.columns)

print("Đã chuẩn bị xong X_scaled, y_class (Phân lớp) và y_reg (Hồi quy)!")

Đã chuẩn bị xong X_scaled, y_class (Phân lớp) và y_reg (Hồi quy)!


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Chia tập Train/Test (Có stratify cho phân lớp)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# 2. Khởi tạo và huấn luyện Hồi quy Logistic
# Vẫn dùng class_weight='balanced' để hỗ trợ nhóm thiểu số (người mắc bệnh)
log_reg = LogisticRegression(class_weight='balanced', random_state=42)
log_reg.fit(X_train_c, y_train_c)

# 3. Đánh giá
y_pred_c = log_reg.predict(X_test_c)
acc_log = accuracy_score(y_test_c, y_pred_c)

print(f"=== KẾT QUẢ LOGISTIC REGRESSION (PHÂN LỚP) ===")
print(f"Độ chính xác (Accuracy): {acc_log * 100:.2f}%\n")
print(classification_report(y_test_c, y_pred_c))

# Xem đặc trưng nào ảnh hưởng nhất (dựa trên trọng số Coefficients)
coef_df = pd.DataFrame({'Feature': X_scaled.columns, 'Weight': log_reg.coef_[0]})
coef_df['Abs_Weight'] = coef_df['Weight'].abs()
coef_df = coef_df.sort_values(by='Abs_Weight', ascending=False)
print("\nTop 3 đặc trưng ảnh hưởng mạnh nhất (theo Hồi quy Logistic):")
print(coef_df[['Feature', 'Weight']].head(3))

=== KẾT QUẢ LOGISTIC REGRESSION (PHÂN LỚP) ===
Độ chính xác (Accuracy): 83.33%

              precision    recall  f1-score   support

           0       0.95      0.85      0.89        65
           1       0.50      0.77      0.61        13

    accuracy                           0.83        78
   macro avg       0.72      0.81      0.75        78
weighted avg       0.87      0.83      0.85        78


Top 3 đặc trưng ảnh hưởng mạnh nhất (theo Hồi quy Logistic):
    Feature    Weight
1  stab.glu  1.976885
9     waist  0.687790
4       age  0.681209
